# Sudo Security

## The `root` account 

On macOS, you can use different accounts to log in. Those accounts can be `standard accounts` or `admin accounts`. There are some things that you can do with an admin account that you can't do with a standard account. For example, only someone with an admin account can create an account for another user.

On some operating systems derived from Unix, it is possible to log in as `root` user who is even higher up the permissions hierarchy than an admin user. In most operating systems, the root can do anything. On macOS, no one can log in as `root`. There, root is an identity that you can take on only briefly and when you do, you are not all powerful. For example, there are directories that you cannot write to because they are controlled by Apple. 

In the terminal, you can take on the permissions of the root user by prefacing a command with `sudo`, which stands for &ldquo;substitute user do.&rdquo; 

For example, you can run the shell command 
    
    cat filename 

as yourself, using your own permissions. Or you can run this command as the `root` by entering  

    sudo cat filename

But if you use the `sudo` preface, you'll have to enter your log in password. 

## The `sudo` command 

The default in macOS is that all admin users are able to run `sudo`. Almost everyone logs in as an admin user, so almost everyone can use `sudo`. 

From these two facts, you might think that having the `sudo` command makes no difference to security. If everyone can act like the root user, what difference does it make? Why not just give the root permissions to all admin users? 

In fact, the `sudo` command is an essential and very important part of the macOS security model. It limits what you can do because that is the way to limit malacious software that is pretending to be you. 

One of the fundamental security weaknesses of Windows is that it has no such command. It is much harder to limit an admin user. As a result, malware that pretends to be an admin user is very powerful. 

The way that the `sudo` limits you is by forcing you to authenticate before the command that it prefaces will run. In the base configuration, the way you authenticate is by typing in your login password into a form that the OS presents to you. If you walk away from your machine while you are logged in, this requirement would stymie a bad actor who takes over and tries to run a `sudo` command. More importantly, it will stymie any malware that is running on your computer because software can't type in anything from the keyboard. 

## The threat 

To keep the roles straight, let's use some names. Suppose that Bob is a software developer with an admin account on his Mac. Mallory has written some malware, a Python script called `harden_ssh.py`. Mallory includes this script in a library he calls `Security_Tools` that he publishes on `https://pypi.org`. 

If Mallory can persuade Bob to run this script on Bob's computer, it will look for a `.ssh` directory in Bob's user's home directory. If it finds one, it tries to read any secret keys that are stored there. It also reads the `config` file to learn the website where Bob uses each key. The script might then use one of the many possible methods to send what it learns to a website that Mallory controls. Finally, the script might try to use git and ssh to submit Mallory's malicious `harden_ssh.py` file to any repositories where Bob is a contributor. 

You should be able to convince yourself that it would be very easy to write such a script. It would take a little bit of social engineering to get Bob to install Mallory's `Security_Tools` library and run the `harden_ssh.py` script, but if Mallory targets 1000 people, it is not hard to believe that at least some of them run the script. 

If Bob runs the script, it runs with Bob's privileges. The permissions on his ssh secret keys may say that only Bob can read these files. In practice, this means that any code that runs while Bob is logged in can also read them, including the code that Mallory writes. 

## How `sudo` helps

Suppose that Bob has configured SSH so that it makes a connection only if the command is prefaced with `sudo`. If Mallory's script includes a `git push` command, it will try to use SSH to make the connection to Github. Now, Bob will see a request for him to approve the `sudo` command by authenticating. Bob knows that he did not run run a `sudo` command or any git command that needs `sudo` so he does not approve it and immediately starts looking for the malware that must have run such a command.  

This stops the malware from using git and ssh to propagate via Bob code repository.

The `sudo` command is helpful here because it requires an action that can't be implemented by software. A person at the keyboard has to enter the keystrokes for Bob's login password. Apple has been careful to ensure that Mallory's Python script can't simulate the entry of the password. As we will see, Apple has also taken steps that make prevent Mallory's script from using a keylogger to record Bob's keystrokes and discover Bob's login password. But as we will see, we need to be aware of the possibility that any malware that runs could deploy a keylogger. 

## Before we start

### With macOS defaults
- You can't run sudo from JupyterLab;
- You can run a sudo command from the JupyterLab Terminal BUT YOU SHOULD NOT; 
- You can run a sudo command from the macOS Terminal BUT YOU SHOULD NOT DO SO UNTIL YOU TURN ON Secure Keyboard Entry in Terminal; 


### Touch ID is great

Shifting to a biometric touch is one of those rare cases where something that is more secure is also more convenient. Of course, you do have to set up TouchID with macOS by registering at least one fingerprint. And you have to be using macOS hardware that has a touch sensor. 


### Once TouchID is set as your default way to authorize `sudo` commands:

- You can safely run sudo commands from a notebook in JupyterLab or from the JupyterLab terminal;
- You will not need to keep the Secure Keyboard Entry option turned on in Terminal;
- In fact, it will be more convenient for you if you turn off Secure Keyboard Entry. 

### What we will do

To follow these instructions you will need to: 

##### 1. Put a file called `sudo-local` into `/etc/pam.d` This will turn on Touch ID for approving sudo requests. 

##### 2. Put a command called `sudo-ssh` into your `/usr/local/bin` folder. This will use sudo every time git uses SSH. 

##### 3. Change your .gitconfig file to make sure git uses `sudo-ssh`

##### 5. Copy files to `/etc/sudoers.d`

##### 6. Change the permissions for the secret key that connects to Github  
